# 02 · Training runs (Passes 1 to 3)

This notebook runs a **list of training runs** and saves each run's scores and predictions. The code lives in the GitHub repo (`src/gainsplit/`); this notebook only fetches it, checks it, picks a list of runs and starts them.

**Before you run, in the right-hand panel:**
- **Accelerator: GPU T4 x2.** Two GPUs, so two runs train at the same time.
- **Internet: on.** Needed to fetch the code from GitHub and BERT from Hugging Face (public, no token needed).
- **Input:** add your private dataset **`gainsplit-processed`**.

**Saving results.** Everything goes to `/kaggle/working/results/<plan>/`. In an interactive session, Kaggle deletes that folder when the session ends, so download the zip from the last cell before closing. A "Save & Run All (Commit)" run keeps its output automatically.

**Plans** (set `PLAN` in step 4): `lr_tuning` (done), `w_tuning` and `gan_pilot` (settings chosen on dev only), `session2` (= `w_tuning` + `gan_pilot`, 12 runs), `pass1` (the main 2x2; needs the chosen helper weight set in `plans.py` first).

## 1. Get the code
Clones the public repo into `/tmp` (outside the output folder, so code and results stay apart) and prints the **commit**. Every run records this commit in its `result.json`, so each number in the paper can be traced to the exact code that made it.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")

COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Smoke test (about a minute)
Runs the repo's tests: a tiny random BERT trains for a few steps on made-up data. It proves the pieces fit together (loading, training, scoring, saving, skipping finished runs, the two-GPU runner) **before** any GPU time is spent on real runs. It says nothing about accuracy. If anything fails, stop here and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Check the GPUs and the data
Finds `gainsplit-processed` under `/kaggle/input` and checks every file against the fingerprints (SHA-256) in `manifest.json`. A mismatch means a file changed and the runs would not match the numbers recorded in the vault.

In [ ]:
import torch
from gainsplit.data import find_data_dir, verify

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DATA_DIR = find_data_dir("/kaggle/input")
manifest = verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
print("made with:", manifest["settings"])

## 4. Choose the list of runs
Set `PLAN` to one of the plans named at the top. The cell below prints every run of the chosen plan: arm, labeled share, seed, learning rate, helper weight and whether the **test** set is scored (tuning plans score **dev only**; the test set stays untouched while settings are chosen).

- `w_tuning`: arm **A3** (BERT + sentiment/emotion helper heads) at **10%** labels, helper weight w in **0.1, 0.3, 1.0**, seeds **0, 1, 2** = 9 runs.
- `gan_pilot`: arm **A4** (GAN-BERT) at **10%** labels, seeds **0, 1, 2** = 3 runs.
- `session2` = both of the above (12 runs).
- `pass1`: the main 2x2 with **test scored**: **A1** at 1, 10, 100% and **A3, A4, A5** at 1, 10%, seeds **0-4** = 45 runs (lr 5e-5, helper weight 0.3, both chosen on dev). Best run as **Save & Run All (Commit)**: about an hour.
- `fgm_tuning`: arm **A7** (BERT + an adversarial push on the word embeddings, "FGM") at **10%** labels, epsilon (size of the push) in **0.5, 1, 2**, seeds **0, 1, 2** = 9 runs, dev only.
- `pass2a` (**104 runs**, done 2026-11-10; about 3.5 GPU-hours, so about 1.7 hours on two GPUs): the controls that explain *where* the Pass 1 gain comes from, plus the budget curve. Test scored, except the 9 `fgm_tuning` runs.
  - **A1, A3, A4, A5** at **5% and 30%** labels, seeds 0-4 (40 runs): fills in the budget curve of the 2x2.
  - **A2** (helper heads trained on the labeled rows only: is the gain just the extra unlabeled text?), **A8** (helper labels shuffled across headlines: is the gain the label content?) and **A6** (self-training: a plain model labels the unlabeled pool, and a fresh model learns from the confident labels) at **1% and 10%**, seeds 0-4 (30 runs).
  - **`fgm_tuning`** (9 runs).
  - **A0** (TF-IDF + logistic regression, a weak floor) at 1, 5, 10, 30, 100%, seeds 0-4 (25 runs). It needs no GPU: it runs on the CPU in the background while the GPUs train.
  - Every arm keeps the same number of training steps at a given budget, so the comparisons stay fair. The longest runs (30%) go first.
- `pass2b` (10 runs, done; about 15 min on two GPUs including setup): **A7** at 1% and 10%, seeds 0-4, test scored, with `PASS2_FGM_EPSILON = 0.5` in `plans.py` (chosen from `fgm_tuning` on dev).
- `pass3` (**105 runs**, **the default**; about 2.7 GPU-hours, so about 1.4 hours on two GPUs): helper-label quality and outlet-style shortcuts. Test scored, and every run also scores the cue-masked and random-masked test sets. Needs `helper_sent_alt.csv`, `headlines_test_cuemask.csv`, `headlines_test_randmask.csv`, `isarcasm_train.csv` and `manifest_pass3.json` in the dataset (made by `01b_pass3_data`).
  - **Helper-label variants** (70 runs): **A3** at 10% and **A2** at 1% with the helper labels changed: `soft` (teacher probabilities instead of hard labels), `flip25` / `flip50` (a quarter / half of the labels changed to a wrong class), `vader` and `teacher2` (another sentiment source), `sentonly` / `emoonly` (only one helper head trained). Seeds 0-4.
  - **A9** (A2 with shuffled helper labels) at 1% (5 runs).
  - **`mask`** (25 runs): A1, A3, A4, A6 and A7 at 10%, unchanged except that the masked test sets are scored too, so they repeat the earlier runs under a new name.
  - **`isarctrain`** (5 runs): A1 trained on iSarcasmEval (10% of it held out as dev) and scored on iSarcasmEval test and Headlines test.


In [ ]:
from gainsplit.plans import PLANS

PLAN = "pass3"
OUT_DIR = f"/kaggle/working/results/{PLAN}"
configs = PLANS[PLAN]()
for c in configs:
    print(f"{c.run_id:26s} arm={c.arm} labels={c.budget}% seed={c.seed} lr={c.lr:g} w={c.helper_weight:g} eps={c.fgm_epsilon:g} variant={c.helper_variant or '-'} train={c.train_data}  "
          f"steps rule: {c.passes} passes over L, min {c.min_steps}  test scored: {c.eval_test}")
print(len(configs), "runs")

## 5. Run
Starts one run per GPU; when a GPU frees up, the next run starts on it. Each run's full log is in `results/<plan>/logs/<run_id>.log`. If the session stops halfway, just run this cell again: finished runs are skipped.

In [ ]:
from gainsplit.runner import run_all

table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
table

## 6. Read the result
Mean and spread (std) over seeds, per arm, learning rate and helper weight. Dev macro-F1 is the main column.

**`w_tuning` (A3):** pick the w with the best mean dev macro-F1. If two values are within about one std, prefer the smaller w. A3 should land near A1's dev macro-F1 at 10% labels (about 0.867); far below 0.8 means something is wrong, so stop and check.

**`gan_pilot` (A4):** dev macro-F1 should be near A1's 0.867 at 10% labels. The second table shows the last training-log point of each GAN run: `p_fake_real` (the discriminator's probability of "fake" on real vectors) should stay well below `p_fake_fake` (on generated vectors). If both sit near 0.5 (the discriminator is not learning) or `p_fake_real` goes to 1 and `p_fake_fake` to 0 (it flips), the GAN is not training properly; report it, do not tune on it.

**`pass2a`:** compare, per budget, A3 against A2 (value of the unlabeled rows), A3 against A8 (value of the helper-label content; A8 should fall back to about A1) and A1 against A0 (A0 is only a floor). For A6 the last table shows how many pseudo-labels were kept; if it is 0, that run fell back to the labeled rows only. For `fgm_tuning`, pick the epsilon with the best mean dev macro-F1 (tie within about one std: the smaller one).

In [ ]:
import json
from pathlib import Path

# one row per arm, label budget and setting; mean and std over seeds
keys = ["arm", "budget", "lr", "helper_weight", "fgm_epsilon"]
cols = [c for c in ["dev_macro_f1", "test_macro_f1", "test_accuracy", "test_roc_auc", "isarc_test_f1_sarcastic",
                    "dev_accuracy", "dev_roc_auc", "minutes"] if c in table.columns]
summary = table.groupby(keys)[cols].agg(["mean", "std"]).round(4)
display(summary)
print("runs per group:"); print(table.groupby(keys).size().to_string())

# GAN runs: last point of the training curve (loss terms and p_fake on real / generated vectors)
rows = []
for f in sorted(Path(OUT_DIR).glob("runs/*/result.json")):
    r = json.loads(f.read_text())
    if r["config"]["arm"] in ("A4", "A5") and r.get("train_log"):
        rows.append({"run_id": r["run_id"], **r["train_log"][-1]})
if rows:
    import pandas as pd
    display(pd.DataFrame(rows).round(4))

# A6: how many unlabeled headlines the teacher labeled with confidence >= 0.9, and how many were kept
# after balancing the two classes (class 0 = not sarcastic, class 1 = sarcastic)
rows = []
for f in sorted(Path(OUT_DIR).glob("runs/*/result.json")):
    r = json.loads(f.read_text())
    if r["config"]["arm"] == "A6" and r.get("pseudo"):
        ps = r["pseudo"]
        rows.append({"run_id": r["run_id"], "unlabeled": ps["n_unlabeled"], "confident": ps["n_confident"],
                     "confident_by_class": ps["confident_class_counts"], "kept": ps["n_kept"],
                     "kept_by_class": ps["kept_class_counts"], "L only (fallback)": ps["fallback_l_only"],
                     "teacher dev macro-F1": ps["teacher_dev_macro_f1"]})
if rows:
    import pandas as pd
    display(pd.DataFrame(rows))

## 7. Download
Zips the results folder. In an interactive session, download `results_<plan>.zip` from the Output panel before you close the session, and keep it in `results/` in your local repo.

In [ ]:
import shutil
zip_path = shutil.make_archive(f"/kaggle/working/results_{PLAN}", "zip", OUT_DIR)
print("download:", zip_path)